# Quantum Table Lookups via Shortest Path

*Dmytro Fedoriaka, 2025*


**TL;DR:** Build a circuit for quantum table lookup with minimum number of Toffoli gates by reducing this problem to finding shortest path in a Cayley graph.


### Problem statement

**Table lookup problem**. Given quantum register of $m$ qubits storing quantum number $i$, target register of $n$ qubits initialized to zeros, and classical lookup table $L = [L[0], \dots L[2^m-1]]$ of $n$-bit unsigned integers, implement a circuit that writes $L[i]$ into the target register:

$$U \ket{i} \ket{0} = \ket{i} \ket{L[i]}$$

One known solution is presented in [1] and  implemented by me [here](https://github.com/fedimser/quant-arith-re/blob/main/lib/src/QuantumArithmetic/TableFunctions.qs). This solution uses $O(2^m)$ gates, including $O(2^m)$ CCNOT gates.


### Reduction to path finding

First, let's assume that target register has only one qubit, so lookup table contains only ones and zeros. If we can solve this problem, we can solve general problem by doing lookup separately for each target qubit, multiplying complexity by $n$. Now our lookup table $T$ is a binary string of length $M=2^m$.

Next, let's assume that $T$ is balanced, that is it has $M/2$ zeros and $M/2$ ones. If it's not balanced, we can also add an extra address qubit and pad $T$ with zeros and ones to make it balanced, see details in section "Efficient padding".

Now, let's try to build a circuit implementing a unitary matrix $U$ acting on $m$ qubits, such that for all $i=0, \dots M-1$:

$$U \ket{i} = \ket{?,?,\dots, T_i}.$$

Here "?" means that value of that qubit is not important, but it's either 0 or 1. This circuit is illustrated in this figure:

<img src="fig1.jpg" width="300">

If we can implement this circuit, we can solve the original lookup problem. One of way of doing that, using uncomputations is shown this figure:

<img src="fig2.jpg" width="600">

Let U be permutation matrix implementing permutation $P$ (that is, $U_{ij}=[i=P_j]$), then:

$$  
  \begin{cases} 
    P_i \in [0, M/2), \text{if}~ T_i=0 \\
    P_i \in [M/2, M), \text{if}~ T_i=1
  \end{cases}
$$

What is action of permutation P on arbitrary string s? $P(s)$ is a string made of the same characters as in $s$, but permuted according to $P$, such that $P(s)_i = s_{P_i}$.

Then 

$$P(T)=S_0 = 00 \dots 0 11 \dots 1,$$

where $S_0$ is the string of length $M$ consisting of $M/2$ zeros followed by $M/2$ ones.

So, we are looking for permutation P, such that $P(T)=S_0$.

Now, let's say that $U$ is a composition of the following gates in some order, maybe with repeats:


* X gates. There are $m$ of them.
* CNOT gates. There are $m(m-1)$ of them.
* CCNOT (aka Toffoli) gates. There are $m(m-1)(m-2)/2$ of them (divide by 2 because order of control qubits doesn't matter).

Each of these gates acts on register as a permutation, and it's easy to compute all these permutations explicitly. Call set of all these permutations $\mathcal{P}$. Note that all these permutations are inverse of themselves.

Now consider the following undirected graph. Vertices are balanced binary strings of length $M$. There are $C_{M}^{M/2} = \frac{M!}{((M/2)!)^2} \sim \frac{2^M}{\sqrt{\pi M}}$ of them. Two vertices are connected by an edge, if one can be transformed to another by applying any permutation from $\mathcal{P}$.

Then the problem is to find a shortest path between vertices $S_0$ and $T$ in this graph. After we have found this path, let's write down permutations corresponding to edges on the path, and then the correspodning gates --- composing these gates will give us matrix U.

If graph is unweighted, we consider X, CNOT and CCNOT to have the same weight. If we consider CCNOT to be much more expensive than X and CNOT, we can say that weight of X and CNOT is 0, and weight of CCNOT is 1.

This problem can be solved with straightforward path finding algorithm like BFS for small $m$, perhaps up to $m=5$. But for $m=5$ graph has $6.01 \cdot 10^8$ vertices, and for $m=6$ it has $1.8 \cdot 10^{18}$ vertices, so even linear (in terms of graph size) algorithms become infeasible.

However, there is a research on finding short (but not necessarily shortest) paths in huge graphs like these, such as [2]. My proposal is to use such algorithms for this problem.

### Efficient padding

In general case, lookup table is not necessarily a power of two, so we don't necessarily need to have an extra address qubit. Also, when we pad, we have a lot of freedom of what to pad with. We only have a fixed number of ones and zeros to add, but order doesn't matter, so we can pick the order that would allow for the most optimal circuit. In terms of path finding in graph this means that we are looking path from $S_0$ not to just one fixed vertex, but to any vertex in certain terminal set.

Let $M_0$ --- the initial size of lookup table. Let $m_0 = \lceil \log_2(M_0) \rceil$. Let $c_0=|T|_0$ --- number of zeros in T, $c_1=|T|_1$ --- number of ones in T. Then there are two cases.

If $c_0 \le 2^{m_0-1}, c_1 \le 2^{m_0-1}$, then $m=m_0$, we need to pad with $p_0=2^{m-1}-c_0$ zeros and $p_1=2^{m-1}-c_1$ ones. We pad initial $T$ with $2^m - M_0$ characters "?", and then expand this string to $C_{2^m - M_0}^{p_0+p_1}$ possible strings by inserting zeros and ones in places of question marks in all possible ways.

If we can't do this, then we pick $m=m_0+1$, and everything else is the same.

The idea here is that when set of terminal strings is big, the shortest path to any of them would be shorter than the shortest path to some arbitrarily chosen string from this set.

Note that with this approach, we are looking for path from $S_0$ to $T$, so to get matrix $U$ we will need to reverse the order of gates.

### Questions and discussion

* The most important result that I would like to try to get from this, is a theoretical proof that table lookup can be done faster than $O(2^m)$ (in terms of circuit depth, i.e. number of gates). If that can not be done in general case, can this be done "in average case", e.g. for a random circuit?
* Here we assume weights $w(X)=w(CNOT)=0$, $w(CCNOT)=1$. In reality all these should be non-zero but different, and values depend on hardware (and also what we optimize for), so ideally solution should be able to work with arbitrary weighted graphs, not just 0-1-graphs, but in principle it's not important. Good starting point to get relative cost of CCNOT gate is [3], where different implementations of CCNOT gate are presented (see fig. 3).
* Are we losing optimality by restricting to permutation gates? That is, if we allowed to consider other, non-permutation, gates (like T gate), would we get much faster circuit? My claim is that it would be still the same asymptotically.

### Experiment

Let's do a simple experiment to validate the approach described above.

For $m=2,3,4,5$ I am going to:
* Generate a circuit for one-bit table lookup for random tables of size $2^m$ using the $O(2^m)$ algorithm, and compute number of X, CNOT, CCNOT gates.
* Then do the same, but use the proposed reduction to path finding. For path finding, we'll use meet-in-the-midle algorithm from the [CayleyPy](https://github.com/cayleypy/cayleypy) library. We repeat this 100 times and compute maximum and average number of gates.

See results in [this public Kaggle notebook](https://www.kaggle.com/code/fedimser/quantum-lookups-with-cayleypy).

We see that for m=5, we reduced number of CCNOT gates from 60 to 10.4 on average.

### Further work
* Implement support for unbalanced lookup tables (see "Efficient padding" section).
* Use different cost for X, CNOT, CCNOT gates (can start with model where X,CNOT have cost 0, XXNOT has cost 1).
* Use machine learning (like in [2]) with beam seacrh, see for what $m$ this will work.
* Compare with AlphaTensor results.

### References

[1] R. Babbush, C. Gidney, D. W. Berry, N. Wiebe, J. McClean, A. Paler, A. Fowler, and
H. Neven, “Encoding electronic spectra in quantum circuits with linear T complexity,” Physical
Review X 8 (2018) no. 4, 041015.

[2] A.Chervov et al, “Cayleypy RL: Pathfinding and reinforcement learning on cayley graphs,” 2025.
https://arxiv.org/abs/2502.18663.

[3] S. Wang, A. Baksi, and A. Chattopadhyay, “A higher radix architecture for quantum
carry-lookahead adder,” Scientific Reports 13 (2023) no. 1, 16338. [Link](https://www.researchgate.net/publication/374263517_A_Higher_radix_architecture_for_quantum_carry-lookahead_adder).